# FOCUS-3D Fine-tuning Demo

This notebook demonstrates how to fine-tune FOCUS-3D using user-curated 3D image-label pairs.

The workflow is:

1. Prepare paired 3D images and labels.
2. Crop them into training patches.
3. Save patches into `imagesTr/` and `labelsTr/`.
4. Fine-tune FOCUS-3D from an existing checkpoint.

Input labels should be 3D instance label maps:

- background = 0
- each cell instance = a positive integer ID

In [ ]:
# ============================================================
# Cell 1. Environment and imports
# ============================================================

import json
import os
import platform
import random
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile
from tqdm.auto import tqdm

# ------------------------------------------------------------
# GPU setting
# ------------------------------------------------------------
# For single-GPU notebook training, use one GPU, for example "0".
# For multi-GPU command-line training, use "0,1,2,3".
#
# If you change CUDA_VISIBLE_DEVICES, restart the notebook kernel.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------
# This notebook is assumed to be located at:
#   focus3d/notebooks/02_finetune_from_patches.ipynb
#
# Backend code is located at:
#   focus3d/src/focus3d/segmentation/FOCUS3D
# ------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
FOCUS3D_BACKEND_DIR = (
    PROJECT_ROOT / 'src' / 'focus3d' / 'segmentation' / 'FOCUS3D'
).resolve()

if str(FOCUS3D_BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(FOCUS3D_BACKEND_DIR))

from train_net_3d import (  # noqa: E402
    build_cfg,
    run_train,
    run_train_distributed,
)

print('FOCUS-3D fine-tuning notebook is ready.')
print('CUDA_VISIBLE_DEVICES =', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('Project root:', PROJECT_ROOT)
print('Backend path:', FOCUS3D_BACKEND_DIR)

## User settings

Modify the following paths and training parameters.

You need paired image and label volumes. The recommended folder structure is:

```text
example/finetune_data/
├── images/
│   ├── volume_001.tif
│   └── volume_002.tif
└── labels/
    ├── volume_001.tif
    └── volume_002.tif

In [ ]:
# ============================================================
# Cell 2. User-editable settings
# ============================================================

# ------------------------------------------------------------
# Input image/label folders
# ------------------------------------------------------------
SOURCE_IMAGE_DIR = PROJECT_ROOT / 'example' / 'finetune_data' / 'images'
SOURCE_LABEL_DIR = PROJECT_ROOT / 'example' / 'finetune_data' / 'labels'

# ------------------------------------------------------------
# Patch dataset output
# ------------------------------------------------------------
PATCH_DATA_DIR = PROJECT_ROOT / 'example' / 'finetune_patches'
PATCH_IMAGE_DIR = PATCH_DATA_DIR / 'imagesTr'
PATCH_LABEL_DIR = PATCH_DATA_DIR / 'labelsTr'

# If True, existing patch data will be removed and regenerated.
RECREATE_PATCHES = True

# If you already have imagesTr/labelsTr, set this to False.
RUN_PATCH_EXPORT = True

# ------------------------------------------------------------
# Model config and checkpoints
# ------------------------------------------------------------
CONFIG_FILE = FOCUS3D_BACKEND_DIR / 'configs' / '3d_test.yaml'

# Full segmentation checkpoint used as fine-tuning initialization.
# This is usually model_final.pth.
INIT_SEG_CHECKPOINT = FOCUS3D_BACKEND_DIR / 'model_final.pth'

# MAE pretraining checkpoint used by the backbone.
# If unavailable, set this to "".
MAE_PRETRAINED = (
    FOCUS3D_BACKEND_DIR / 'MAE_checkpoint_0510' / 'checkpoint-399.pth'
)

# Fine-tuning output folder.
OUTPUT_DIR = PROJECT_ROOT / 'example' / 'finetune_output'

# ------------------------------------------------------------
# Patch extraction parameters
# ------------------------------------------------------------
PATCH_SIZE = [32, 96, 96]  # Z, Y, X
PATCH_STRIDE = [24, 64, 64]  # Z, Y, X

# Skip patches with too little annotated foreground.
MIN_LABEL_VOXELS = 64

# Skip patches whose raw image is nearly empty.
# Set to None to disable raw-intensity filtering.
BACKGROUND_THRESHOLD = None

# Limit patches per volume.
# Set to None to keep all valid patches.
MAX_PATCHES_PER_VOLUME = None

RANDOM_SEED = 2026

# ------------------------------------------------------------
# Fine-tuning parameters
# ------------------------------------------------------------
NUM_GPUS = 1

# Total batch size across all GPUs.
# For example, if NUM_GPUS=4 and IMS_PER_BATCH=16,
# each GPU processes approximately 4 patches.
IMS_PER_BATCH = 4

BASE_LR = 1e-4
MAX_ITER = 2000
WARMUP_ITERS = 100
CHECKPOINT_PERIOD = 500
EVAL_PERIOD = 0

# Resume from OUTPUT_DIR if an existing checkpoint is found.
RESUME = False

# Use AMP mixed precision.
AMP_ENABLED = True

# ------------------------------------------------------------
# Print settings
# ------------------------------------------------------------
settings = {
    'SOURCE_IMAGE_DIR': str(SOURCE_IMAGE_DIR),
    'SOURCE_LABEL_DIR': str(SOURCE_LABEL_DIR),
    'PATCH_DATA_DIR': str(PATCH_DATA_DIR),
    'CONFIG_FILE': str(CONFIG_FILE),
    'INIT_SEG_CHECKPOINT': str(INIT_SEG_CHECKPOINT),
    'MAE_PRETRAINED': str(MAE_PRETRAINED),
    'OUTPUT_DIR': str(OUTPUT_DIR),
    'PATCH_SIZE': PATCH_SIZE,
    'PATCH_STRIDE': PATCH_STRIDE,
    'MIN_LABEL_VOXELS': MIN_LABEL_VOXELS,
    'BACKGROUND_THRESHOLD': BACKGROUND_THRESHOLD,
    'MAX_PATCHES_PER_VOLUME': MAX_PATCHES_PER_VOLUME,
    'NUM_GPUS': NUM_GPUS,
    'IMS_PER_BATCH': IMS_PER_BATCH,
    'BASE_LR': BASE_LR,
    'MAX_ITER': MAX_ITER,
}

print(json.dumps(settings, indent=2))

In [ ]:
# ============================================================
# Cell 3. Check paths
# ============================================================


def check_exists(path, name, allow_empty_string=False):
    if allow_empty_string and str(path).strip() == '':
        print(f'{name}: not used')
        return None

    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'{name} not found: {path}')

    print(f'{name}: {path}')
    return path


if RUN_PATCH_EXPORT:
    check_exists(SOURCE_IMAGE_DIR, 'Source image folder')
    check_exists(SOURCE_LABEL_DIR, 'Source label folder')

check_exists(CONFIG_FILE, 'Config file')
check_exists(INIT_SEG_CHECKPOINT, 'Initial segmentation checkpoint')

if str(MAE_PRETRAINED).strip() != '':
    if Path(MAE_PRETRAINED).exists():
        print('MAE pretrained checkpoint:', MAE_PRETRAINED)
    else:
        print('Warning: MAE pretrained checkpoint not found.')
        print(
            'It will be passed to config, but training may skip it if the file is unavailable.'
        )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output folder:', OUTPUT_DIR)

## Patch extraction

This section converts full 3D volumes into fixed-size training patches.

The output structure will be:

```text
finetune_patches/
├── imagesTr/
│   ├── volume_001_z0000_y0000_x0000.tif
│   └── ...
└── labelsTr/
    ├── volume_001_z0000_y0000_x0000.tif
    └── ...

In [ ]:
# ============================================================
# Cell 4. Patch extraction helper functions
# ============================================================


def read_volume(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix in ['.tif', '.tiff']:
        arr = tifffile.imread(str(path))
    elif suffix == '.zarr':
        import zarr

        arr = np.asarray(zarr.open(str(path), mode='r'))
    else:
        raise ValueError(f'Unsupported file format: {path}')

    arr = np.squeeze(np.asarray(arr))

    if arr.ndim != 3:
        raise ValueError(
            f'Expected a 3D volume, got shape {arr.shape}: {path}'
        )

    return arr


def save_tif(path, arr):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tifffile.imwrite(str(path), arr)


def compute_axis_starts(length, patch_len, stride_len):
    """
    Generate patch starts along one axis.

    Rules:
    - always include 0
    - step by stride
    - always include the last start so the tail is covered
    """
    length = int(length)
    patch_len = int(patch_len)
    stride_len = int(stride_len)

    if length <= patch_len:
        return [0]

    starts = list(range(0, length - patch_len + 1, stride_len))
    last_start = length - patch_len

    if starts[-1] != last_start:
        starts.append(last_start)

    return starts


def generate_patch_coords(volume_shape, patch_size, stride):
    z, y, x = volume_shape
    pz, py, px = patch_size
    sz, sy, sx = stride

    z_starts = compute_axis_starts(z, pz, sz)
    y_starts = compute_axis_starts(y, py, sy)
    x_starts = compute_axis_starts(x, px, sx)

    coords = []
    for z0 in z_starts:
        for y0 in y_starts:
            for x0 in x_starts:
                coords.append((z0, y0, x0))

    return coords


def crop_with_padding(volume, start, patch_size, pad_mode):
    """
    Crop a 3D patch. If the input volume is smaller than the patch,
    pad the patch at the end of each axis.
    """
    z0, y0, x0 = start
    pz, py, px = patch_size

    z1 = min(z0 + pz, volume.shape[0])
    y1 = min(y0 + py, volume.shape[1])
    x1 = min(x0 + px, volume.shape[2])

    patch = volume[z0:z1, y0:y1, x0:x1]

    pad_z = pz - patch.shape[0]
    pad_y = py - patch.shape[1]
    pad_x = px - patch.shape[2]

    if pad_z > 0 or pad_y > 0 or pad_x > 0:
        patch = np.pad(
            patch,
            ((0, pad_z), (0, pad_y), (0, pad_x)),
            mode=pad_mode,
        )

    return patch


def relabel_instances(label_patch):
    """
    Relabel positive instance IDs to consecutive IDs within each patch.

    Background remains 0.
    """
    label_patch = np.asarray(label_patch)
    ids = np.unique(label_patch)
    ids = ids[ids > 0]

    if len(ids) == 0:
        return np.zeros_like(label_patch, dtype=np.uint16)

    out = np.zeros_like(label_patch, dtype=np.uint32)

    for new_id, old_id in enumerate(ids, start=1):
        out[label_patch == old_id] = new_id

    if out.max() <= np.iinfo(np.uint16).max:
        return out.astype(np.uint16)

    return out.astype(np.uint32)


def find_paired_files(image_dir, label_dir):
    """
    Match image and label files by file stem.

    Example:
    images/00000.tif
    labels/00000.tif
    """
    image_dir = Path(image_dir)
    label_dir = Path(label_dir)

    image_files = []
    for pat in ['*.tif', '*.tiff', '*.TIF', '*.TIFF', '*.zarr', '*.ZARR']:
        image_files.extend(image_dir.glob(pat))

    label_files = []
    for pat in ['*.tif', '*.tiff', '*.TIF', '*.TIFF', '*.zarr', '*.ZARR']:
        label_files.extend(label_dir.glob(pat))

    image_map = {p.stem: p for p in image_files}
    label_map = {p.stem: p for p in label_files}

    common_stems = sorted(set(image_map.keys()) & set(label_map.keys()))

    if len(common_stems) == 0:
        raise FileNotFoundError(
            'No paired image/label files found. '
            'Image and label files should have the same file stem.'
        )

    pairs = [(image_map[s], label_map[s]) for s in common_stems]
    return pairs


def export_training_patches(
    source_image_dir,
    source_label_dir,
    patch_image_dir,
    patch_label_dir,
    patch_size,
    stride,
    min_label_voxels=64,
    background_threshold=None,
    max_patches_per_volume=None,
    recreate=True,
    seed=2026,
):
    """
    Export paired image/label patches for FOCUS-3D fine-tuning.
    """
    random.seed(seed)
    np.random.seed(seed)

    patch_image_dir = Path(patch_image_dir)
    patch_label_dir = Path(patch_label_dir)

    if recreate:
        if patch_image_dir.exists():
            shutil.rmtree(patch_image_dir)
        if patch_label_dir.exists():
            shutil.rmtree(patch_label_dir)

    patch_image_dir.mkdir(parents=True, exist_ok=True)
    patch_label_dir.mkdir(parents=True, exist_ok=True)

    pairs = find_paired_files(source_image_dir, source_label_dir)

    print(f'Found {len(pairs)} paired image/label volume(s).')
    print('Patch image dir:', patch_image_dir)
    print('Patch label dir:', patch_label_dir)

    summary = []
    total_saved = 0

    for image_path, label_path in tqdm(
        pairs, desc='Export patches', unit='vol'
    ):
        image = read_volume(image_path)
        label = read_volume(label_path)

        if image.shape != label.shape:
            raise ValueError(
                f'Image and label shape mismatch:\n'
                f'Image: {image_path}, shape={image.shape}\n'
                f'Label: {label_path}, shape={label.shape}'
            )

        coords = generate_patch_coords(image.shape, patch_size, stride)

        valid_coords = []
        for coord in coords:
            image_patch = crop_with_padding(
                image,
                coord,
                patch_size,
                pad_mode='reflect',
            )

            label_patch = crop_with_padding(
                label,
                coord,
                patch_size,
                pad_mode='constant',
            )

            n_label_voxels = int(np.count_nonzero(label_patch > 0))

            if n_label_voxels < int(min_label_voxels):
                continue

            if background_threshold is not None and float(
                np.percentile(image_patch, 99.9)
            ) < float(background_threshold):
                continue

            valid_coords.append(coord)

        if max_patches_per_volume is not None and len(valid_coords) > int(
            max_patches_per_volume
        ):
            valid_coords = random.sample(
                valid_coords, int(max_patches_per_volume)
            )
            valid_coords = sorted(valid_coords)

        saved_this_volume = 0

        for coord in valid_coords:
            z0, y0, x0 = coord

            image_patch = crop_with_padding(
                image,
                coord,
                patch_size,
                pad_mode='reflect',
            )

            label_patch = crop_with_padding(
                label,
                coord,
                patch_size,
                pad_mode='constant',
            )

            label_patch = relabel_instances(label_patch)

            patch_name = f'{image_path.stem}_z{z0:04d}_y{y0:04d}_x{x0:04d}.tif'

            save_tif(patch_image_dir / patch_name, image_patch)
            save_tif(patch_label_dir / patch_name, label_patch)

            saved_this_volume += 1
            total_saved += 1

        summary.append(
            {
                'image': str(image_path),
                'label': str(label_path),
                'shape': image.shape,
                'all_patches': len(coords),
                'saved_patches': saved_this_volume,
            }
        )

    print('\nPatch export finished.')
    print('Total saved patches:', total_saved)

    return summary

In [ ]:
# ============================================================
# Cell 5. Run patch extraction
# ============================================================

if RUN_PATCH_EXPORT:
    patch_summary = export_training_patches(
        source_image_dir=SOURCE_IMAGE_DIR,
        source_label_dir=SOURCE_LABEL_DIR,
        patch_image_dir=PATCH_IMAGE_DIR,
        patch_label_dir=PATCH_LABEL_DIR,
        patch_size=PATCH_SIZE,
        stride=PATCH_STRIDE,
        min_label_voxels=MIN_LABEL_VOXELS,
        background_threshold=BACKGROUND_THRESHOLD,
        max_patches_per_volume=MAX_PATCHES_PER_VOLUME,
        recreate=RECREATE_PATCHES,
        seed=RANDOM_SEED,
    )

    print('\nPatch summary:')
    for item in patch_summary:
        print(
            f'- {Path(item["image"]).name}: '
            f'{item["saved_patches"]} / {item["all_patches"]} patches saved'
        )
else:
    print('Patch export skipped. Existing imagesTr/labelsTr will be used.')

if not PATCH_IMAGE_DIR.exists() or not PATCH_LABEL_DIR.exists():
    raise FileNotFoundError('imagesTr/labelsTr folders are missing.')

n_images = len(list(PATCH_IMAGE_DIR.glob('*.tif')))
n_labels = len(list(PATCH_LABEL_DIR.glob('*.tif')))

print('\nTraining patch folders:')
print('Images:', PATCH_IMAGE_DIR, 'count =', n_images)
print('Labels:', PATCH_LABEL_DIR, 'count =', n_labels)

if n_images == 0 or n_labels == 0:
    raise RuntimeError('No training patches found.')

if n_images != n_labels:
    raise RuntimeError(
        'The number of image patches and label patches is different.'
    )

## Preview training patches

This section shows one exported training patch and its label.

This is a quick sanity check before starting fine-tuning.

In [ ]:
# ============================================================
# Cell 6. Preview one training patch
# ============================================================


def normalize_for_display(img2d, p_low=1, p_high=99):
    img2d = np.asarray(img2d, dtype=np.float32)
    lo, hi = np.percentile(img2d, [p_low, p_high])
    if hi <= lo:
        return np.zeros_like(img2d, dtype=np.float32)
    img2d = np.clip(img2d, lo, hi)
    return (img2d - lo) / (hi - lo)


patch_files = sorted(PATCH_IMAGE_DIR.glob('*.tif'))
preview_image_path = patch_files[0]
preview_label_path = PATCH_LABEL_DIR / preview_image_path.name

preview_image = tifffile.imread(str(preview_image_path))
preview_label = tifffile.imread(str(preview_label_path))

z = preview_image.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(normalize_for_display(preview_image[z]), cmap='gray')
axes[0].set_title(f'Image patch, Z={z}')
axes[0].axis('off')

axes[1].imshow(preview_label[z], cmap='nipy_spectral', interpolation='nearest')
axes[1].set_title(f'Label patch, Z={z}')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print('Preview image:', preview_image_path)
print('Preview label:', preview_label_path)
print('Image shape:', preview_image.shape)
print('Label shape:', preview_label.shape)
print('Max label ID:', int(preview_label.max()))

## Build fine-tuning configuration

The original `3d_test.yaml` is used as the base config.

The notebook overrides the most important parameters:

- training image folder;
- training label folder;
- output folder;
- initial segmentation checkpoint;
- MAE pretrained checkpoint;
- patch size;
- batch size;
- learning rate;
- maximum iterations;
- checkpoint period.

For notebook use, single-GPU training is recommended. Multi-GPU training is usually more stable from the command line.

In [ ]:
# ============================================================
# Cell 7. Build config overrides
# ============================================================


def cfg_value(x):
    """
    Convert Python values to strings accepted by Detectron2/YACS opts.
    """
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, str):
        return x
    if isinstance(x, bool):
        return 'True' if x else 'False'
    if isinstance(x, (list, tuple)):
        return str(list(x))
    return str(x)


OPTS = [
    # Dataset paths
    'DATASETS.IMAGE_DIR',
    cfg_value(PATCH_IMAGE_DIR),
    'DATASETS.LABEL_DIR',
    cfg_value(PATCH_LABEL_DIR),
    # Output folder
    'OUTPUT_DIR',
    cfg_value(OUTPUT_DIR),
    # Initial full segmentation checkpoint
    'MODEL.WEIGHTS',
    cfg_value(INIT_SEG_CHECKPOINT),
    # MAE pretrained checkpoint
    'MODEL.BACKBONE.PRETRAINED',
    cfg_value(MAE_PRETRAINED),
    # Input / patch size
    'INPUT.IMAGE_SIZE',
    cfg_value(PATCH_SIZE),
    'INPUT.MIN_SIZE_TRAIN',
    cfg_value(PATCH_SIZE),
    'MODEL.BACKBONE.IMG_SIZE',
    cfg_value(PATCH_SIZE),
    'MODEL.BACKBONE.VIT_ADAPTER.PRETRAIN_SIZE',
    cfg_value(PATCH_SIZE),
    # Training hyperparameters
    'SOLVER.IMS_PER_BATCH',
    cfg_value(IMS_PER_BATCH),
    'SOLVER.BASE_LR',
    cfg_value(BASE_LR),
    'SOLVER.MAX_ITER',
    cfg_value(MAX_ITER),
    'SOLVER.WARMUP_ITERS',
    cfg_value(WARMUP_ITERS),
    'SOLVER.CHECKPOINT_PERIOD',
    cfg_value(CHECKPOINT_PERIOD),
    'SOLVER.AMP.ENABLED',
    cfg_value(AMP_ENABLED),
    # Evaluation
    'TEST.EVAL_PERIOD',
    cfg_value(EVAL_PERIOD),
]

print('Config overrides:')
for k, v in zip(OPTS[0::2], OPTS[1::2], strict=False):
    print(f'{k}: {v}')

In [ ]:
# ============================================================
# Cell 8. Create config object
# ============================================================

cfg = build_cfg(str(CONFIG_FILE), OPTS)

print('Final training config:')
print('DATASETS.IMAGE_DIR:', cfg.DATASETS.IMAGE_DIR)
print('DATASETS.LABEL_DIR:', cfg.DATASETS.LABEL_DIR)
print('OUTPUT_DIR:', cfg.OUTPUT_DIR)
print('MODEL.WEIGHTS:', cfg.MODEL.WEIGHTS)
print('MODEL.BACKBONE.PRETRAINED:', cfg.MODEL.BACKBONE.PRETRAINED)
print('INPUT.IMAGE_SIZE:', cfg.INPUT.IMAGE_SIZE)
print('SOLVER.IMS_PER_BATCH:', cfg.SOLVER.IMS_PER_BATCH)
print('SOLVER.BASE_LR:', cfg.SOLVER.BASE_LR)
print('SOLVER.MAX_ITER:', cfg.SOLVER.MAX_ITER)
print('SOLVER.CHECKPOINT_PERIOD:', cfg.SOLVER.CHECKPOINT_PERIOD)
print('SOLVER.AMP.ENABLED:', cfg.SOLVER.AMP.ENABLED)

Path(cfg.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

## Start fine-tuning

For most notebook users, use `NUM_GPUS = 1`.

For multi-GPU fine-tuning, command-line training is recommended:

```bash
CUDA_VISIBLE_DEVICES=0,1,2,3 python train_net_3d.py \
  --config-file ./configs/3d_test.yaml \
  --num-gpus 4 \
  DATASETS.IMAGE_DIR /path/to/imagesTr \
  DATASETS.LABEL_DIR /path/to/labelsTr \
  OUTPUT_DIR /path/to/output \
  MODEL.WEIGHTS /path/to/model_final.pth

In [ ]:
# ============================================================
# Cell 9. Start fine-tuning
# ============================================================

if NUM_GPUS == 1:
    print('Starting single-GPU fine-tuning inside notebook...')
    print(
        'If you changed CUDA_VISIBLE_DEVICES, restart the kernel before running this cell.'
    )
    train_result = run_train(cfg, resume=RESUME)
else:
    print('Starting distributed fine-tuning from notebook...')
    print('For large multi-GPU jobs, running from terminal is recommended.')
    train_result = run_train_distributed(
        config_file=str(CONFIG_FILE),
        num_gpus=int(NUM_GPUS),
        resume=RESUME,
        opts=OPTS,
        dist_url='auto',
    )

print('Fine-tuning finished.')
print('Training result:', train_result)

In [ ]:
# ============================================================
# Cell 10. Locate checkpoints
# ============================================================

output_dir = Path(OUTPUT_DIR)
checkpoints = sorted(output_dir.glob('*.pth'))

print('Output folder:', output_dir)
print(f'Found {len(checkpoints)} checkpoint file(s).')

for p in checkpoints:
    print(p)

final_ckpt = output_dir / 'model_final.pth'
if final_ckpt.exists():
    print('\nFinal checkpoint:')
    print(final_ckpt)
else:
    print(
        '\nmodel_final.pth was not found yet. Training may have stopped before final saving.'
    )

In [ ]:
# ============================================================
# Cell 11. Save a reusable command-line training script
# ============================================================


def make_train_command():
    opts_text = ' '.join(
        f'"{k}" "{v}"' for k, v in zip(OPTS[0::2], OPTS[1::2], strict=False)
    )

    if platform.system().lower() == 'windows':
        cmd = (
            f'set CUDA_VISIBLE_DEVICES={os.environ.get("CUDA_VISIBLE_DEVICES", "0")} && '
            f'python train_net_3d.py '
            f'--config-file "{CONFIG_FILE}" '
            f'--num-gpus {NUM_GPUS} '
            f'{opts_text}'
        )
        script_path = OUTPUT_DIR / 'run_finetune.bat'
    else:
        cmd = (
            '#!/bin/bash\n\n'
            f'export CUDA_VISIBLE_DEVICES={os.environ.get("CUDA_VISIBLE_DEVICES", "0")}\n\n'
            f'python train_net_3d.py \\\n'
            f'  --config-file "{CONFIG_FILE}" \\\n'
            f'  --num-gpus {NUM_GPUS} \\\n'
            f'  {opts_text}\n'
        )
        script_path = OUTPUT_DIR / 'run_finetune.sh'

    return cmd, script_path


cmd, script_path = make_train_command()

script_path.parent.mkdir(parents=True, exist_ok=True)
script_path.write_text(cmd, encoding='utf-8')

print('Reusable training command saved to:')
print(script_path)

print('\nCommand:')
print(cmd)